In [1]:
# engine abstraction with PAC guarantee example

# needed libraries
import numpy as np
import scipy.special as sp
import time
from itertools import product
import random
import gurobipy as gp
from gurobipy import GRB
from joblib import Parallel, delayed
from scipy.optimize import fsolve
from math import comb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
from scipy.optimize import brentq
from scipy.stats import truncnorm

In [13]:
# choice of N for repetitve scenario
N = 1100
n_dim = 2 # dimension of state set
N_pos = 1000 #100 # N used for computing post in abstraction

In [14]:
# system dynamics
tau = 0.1 #sampling time
eta_x1 = 16e-6  
eta_x = np.array([eta_x1,eta_x1]) # discretization vector
# set of states
a1, a2, b1, b2 = 0.4479, 0.4559, 0.6473, 0.6553
a = np.array([a1, b1])  # lower bounds
b = np.array([a2, b2])  # upper bounds

dim1_hat = np.arange(a1+eta_x1, a2-eta_x1, eta_x1)
dim2_hat = np.arange(b1+eta_x1, b2-eta_x1, eta_x1)
X_ha = np.array(list(product(dim1_hat, dim2_hat))) 
X_haa = np.around(X_ha, 4)

In [15]:
# --- Truncated normal for continuous x_samples ---
def sample_truncnorm(a, b, mean, std, size):
    a_, b_ = (a - mean)/std, (b - mean)/std
    return truncnorm.rvs(a_, b_, loc=mean, scale=std, size=size)

# Example parameters
mean = (a + b)/2
std = (b - a)/4  # roughly cover the interval

x_samples = sample_truncnorm(a, b, mean, std, size=(N, 2))

# --- Gaussian sampling for x_hat_samples from discrete grid X_haa ---
def sample_from_grid_gaussian(X_grid, N, mean_point=0, std_dev=np.sqrt(0.1)):
    """
    X_grid: array of shape (M, dim) representing the discrete grid
    mean_point: point in grid coordinates (index) to center Gaussian around
    std_dev: controls spread over grid indices
    """
    M = len(X_grid)
    if mean_point is None:
        mean_point = M / 2
    # Truncated normal over indices
    a_, b_ = (0 - mean_point)/std_dev, (M-1 - mean_point)/std_dev
    idx = truncnorm.rvs(a_, b_, loc=mean_point, scale=std_dev, size=N).astype(int)
    return X_grid[idx]

x_hat_samples = sample_from_grid_gaussian(X_haa, N, mean_point=len(X_haa)/2, std_dev=len(X_haa)/6)

# --- Combine into pairs ---
X_pairs = list(zip(x_samples, x_hat_samples))
print(len(X_pairs))

1100


In [16]:
# system dynamics as black-box simulator
# dynamics parameter
a = 1.0/3.5;
H = 0.18;
l = 8;
B = 2;
W = 0.25;

def sys_dyn(x_t, u_t):
    u1, u2 = u_t
    x1, x2 = x_t
    psi = a + H*(1 + 1.5*(x1/W-1) - 0.5*(x1/W-1)**3)
    f_xt1 = x1 + tau*(1/l*(psi - x2) + u1)
    f_xt2 = x2 + tau*(1/(4*l*B*B)*(x1 - u2*np.sqrt(x2)))
    nxt = [max(min(f_xt1, a2), a1), max(min(f_xt2, b2), b1)]
    nxt = [round(x, 2) for x in nxt]
    return nxt

def generate_grid_centers(a, b, N_pos):
    dim = len(a)
    # Estimate number of points per dimension (evenly)
    k = int(np.round(N_pos ** (1 / dim)))
    total_points = k ** dim

    # Generate grid centers per dimension
    grid_axes = []
    for ai, bi in zip(a, b):
        step = (bi - ai) / k
        centers = ai + (np.arange(k) + 0.5) * step
        grid_axes.append(centers)

    # Cartesian product of all centers (i.e., subgrid centers)
    all_points = np.array(list(product(*grid_axes)))

    # Trim if more than needed
    if total_points > N_pos:
        all_points = all_points[:N_pos]

    return all_points

# for abstract system
def sys_dyn_hat(y, u):
    y = np.asarray(y)
    X_hat_arra = np.array(X_haa)  
    a, b = y-eta_x/2, y+eta_x/2 
    # sampling N points as subgrid centres for each cell to obtain an estimate of the reachable sets as done in the paper
    sampled_points = generate_grid_centers(a, b, N_pos)

    # Compute successors
    successors = np.array([sys_dyn(x, u) for x in sampled_points])

    # Compute mean and max distance
    m = np.mean(successors, axis=0)
    r = np.max(np.abs(successors - m), axis=0)

    # Find points in X_hat_array within the under-approximation of reachable sets
    mask = np.all(np.abs(X_hat_arra - m) <= (r + eta_x / 2), axis=1)

    return X_hat_arra[mask]

In [17]:
# set of inputs
eta_u1, eta_u2 = 1.0/30, 0.1  
dim_u1 = np.arange(-0.05, 0.05 - eta_u1, eta_u1)
dim_u2 = np.arange(0.5, 0.8 - eta_u2, eta_u2)
U_ha = np.array(list(product(dim_u1, dim_u2)))
U = np.around(U_ha, 2).tolist()
M = len(U)
U_array = np.array(U)
print(M)

ubpr = 10
lbp, ubp = -200, 200
lbpc, ubpc = -200, 200
lbe, ube = -100, 0 

9


#### the SOP

In [18]:
m1 = gp.Model("PAC_ASF")

vartheta = m1.addVar(vtype = GRB.CONTINUOUS, name = "vartheta", lb = lbe, ub = ube)

eps_vars = 1/len(X_pairs) 
# eps_vars = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"eps_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_eps = gp.quicksum(eps_var for eps_var in eps_vars)
# m1.addConstr(sum_eps == 1)

# eps_varsp = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"epsp_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_epsp = gp.quicksum(eps_var for eps_var in eps_varsp)
# m1.addConstr(sum_epsp == 1)
m1.update()

In [19]:
gamma_b = 1e4 
delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
# gamma_b = m1.addVar(vtype = GRB.CONTINUOUS, name = "gamma_b", lb = 10, ub = ubpr)
tau_b = 1.005 #m1.addVar(vtype = GRB.CONTINUOUS, name = "tau_b", lb = 0, ub = ubpr) # this is \rho in the paper
print("delta: ", delta_b)
print("epsilon: ", (delta_b/gamma_b)**0.5)

num_variables = 15  # number of coefficients as a result of the chosen degree of asf
variable_names = [f"lambda_{i}" for i in range(1, num_variables + 1)]
variable_objs = {name_i: m1.addVar(vtype=GRB.CONTINUOUS, name=name_i, lb=lbpc, ub=ubpc) for name_i in variable_names}

def Asf(x, x_hat):
    # Coefficients from the declared variables
    c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15 = variable_objs.values()
    x1, x2, y1, y2 = x[0], x[1], x_hat[0], x_hat[1]
    # Polynomial expression using the variables
    result = (
        c1 + c2*x1 + c3*x2 + c4*y1 + c5*y2 +
        c6*x1**2 + c7*x1*x2 + c8*x1*y1 + c9*x1*y2 +
        c10*x2**2 + c11*x2*y1 + c12*x2*y2 +
        c13*y1**2 + c14*y1*y2 + 
        c15*y2**2
    )
    return result

m1.update()

delta:  2.56e-06
epsilon:  1.6e-05


In [20]:
# Define lambda functions for efficiency
max_abs_diff = lambda x, y: gamma_b * np.max(np.abs(x - y))**2
asf_value = lambda x, y: Asf(x, y)

# First ASF condition
t_init = time.time()
# sum1 = gp.quicksum(eps_vars[l] * (max_abs_diff(x, y) - asf_value(x, y)) for l, (x,y) in enumerate(X_pairs))
sum1 = gp.quicksum(eps_vars * (max_abs_diff(x, y) - asf_value(x, y)) for (x,y) in X_pairs)
m1.addConstr(sum1 <= vartheta)
t_fin = time.time()
diff_t = t_fin - t_init
print(diff_t)
m1.update()

0.07030916213989258


In [21]:
t_init = time.time()
# ASF 2 Enforce sum2 - Asf <= vartheta

def compute_sys_dyn(l, i):
    return l, i, sys_dyn_hat(X_pairs[l][1], U[i]), sys_dyn(X_pairs[l][0], U[i])

# Step 1: Compute system dynamics in parallel
num_jobs = -1  # Use all available cores
dyn_results = Parallel(n_jobs=num_jobs, verbose=2)(
    delayed(compute_sys_dyn)(l, i) for l in range(len(X_pairs)) for i in range(len(U))
)

print("done1", len(dyn_results))
# Step 2: Add constraints sequentially (Gurobi constraint addition must be sequential during parallelization)
for idx, (l, i, dyn_hat_output, dyn_output) in enumerate(dyn_results):
    m1.addConstr(
        gp.quicksum(1/(len(dyn_hat_output)) * asf_value(dyn_output, yn) for yn in dyn_hat_output) # sigma taken uniformly
        - tau_b*asf_value(X_pairs[l][0], X_pairs[l][1]) + (tau_b-1)*delta_b <= vartheta
    )
    # Print progress every 100 iterations
    if (idx + 1) % 100 == 0:
        print('.', end='', flush=True)

t_fin = time.time()
diff_t = t_fin - t_init
m1.update()
print(diff_t)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 345 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done 628 tasks      | elapsed:   10.7s
[Parallel(n_jobs=-1)]: Done 993 tasks      | elapsed:   15.5s
[Parallel(n_jobs=-1)]: Done 1438 tasks      | elapsed:   21.3s
[Parallel(n_jobs=-1)]: Done 1965 tasks      | elapsed:   28.4s
[Parallel(n_jobs=-1)]: Done 2572 tasks      | elapsed:   36.4s
[Parallel(n_jobs=-1)]: Done 3261 tasks      | elapsed:   45.6s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:   55.9s
[Parallel(n_jobs=-1)]: Done 4881 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 5812 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 6825 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 7918 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 9093 tasks      | 

done1 9900
...................................................................................................143.09015011787415


In [22]:
m1.Params.LogToConsole = 0
m1.setParam("NumericFocus", 3)
m1.setParam('NonConvex', 2)

m1.update()
print("Start now")
print("Number of variables:", m1.numVars)
print("Number of constraints:", m1.numConstrs)
t_init = time.time()

m1.setObjective(vartheta, GRB.MINIMIZE)
m1.setParam('Presolve', 0)

m1.optimize()
t_fin = time.time()
diff_t = t_fin - t_init
print("opt_status:", m1.status)

m1.write("engine_with_PAC_Infeasible_most_recent.lp")

# Check if optimization was successful
if m1.status == GRB.OPTIMAL:
    with open('engine_with_PAC_variables_most_recent.txt', 'w') as file:
        for v in m1.getVars():
            file.write(f'{v.VarName} {v.X}\n')
elif m1.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m1.computeIIS()
    m1.write("engine_with_PAC_Infeasible_most_recent.ilp")
    print("IIS written to engine_with_PAC_Infeasible_most_recent.ilp")

    with open("engine_with_PAC_Infeasible_most_recent.txt", "w") as f:
        for c in m1.getConstrs():
            if c.IISConstr:
                f.write(f"Infeasible constraint: {c.ConstrName}\n")
        for v in m1.getVars():
            if v.IISLB or v.IISUB:
                f.write(f"Infeasible bound on variable: {v.VarName}\n")

else:
    print("Optimization was not successful (not optimal or infeasible). Status:", m1.status)

# Count binding constraints
sup_const = [constr for constr in m1.getConstrs() if abs(constr.Slack) < 1e-4]
s_ = len(sup_const)
print(f"Number of constraints within tol: {s_}")
print("Time (in s):", diff_t)

Start now
Number of variables: 16
Number of constraints: 9901
opt_status: 2
Number of constraints within tol: 18
Time (in s): 0.0111846923828125


In [23]:
# Count binding constraints
def prune_constraints_inf_norm(m1, tol=1e-4):
    """
    Iteratively test which constraints are necessary
    based on their effect on the optimal solution.
    Uses infinity norm for solution difference.
    Returns the final set of constraints deemed necessary.
    """
    # Solve original model
    m1.optimize()
    if m1.status != gp.GRB.OPTIMAL:
        raise RuntimeError("Original model is not optimal.")
    
    # Extract original solution sol*
    sol_star = np.array([v.X for v in m1.getVars()])
    
    # Initial set of constraints
    C = list(m1.getConstrs())
    
    necessary_constraints = []
    
    for constr in C:
        # Create a clone of the model
        m_copy = m1.copy()
        
        # Remove the constraint from the copy
        constr_copy = m_copy.getConstrByName(constr.ConstrName)
        m_copy.remove(constr_copy)
        m_copy.update()
        
        # Solve the reduced model
        m_copy.optimize()
        
        if m_copy.status == gp.GRB.OPTIMAL:
            sol_starstar = np.array([v.X for v in m_copy.getVars()])
            diff = np.linalg.norm(sol_star - sol_starstar, ord=np.inf)  # infinity norm
            
            # If the solution shifts a lot, keep the constraint
            if diff > tol:
                necessary_constraints.append(constr)
        # else:
        #     # If infeasible or unbounded without this constraint, it’s definitely necessary
        #     necessary_constraints.append(constr)
    
    return necessary_constraints

# Usage
s_N = len(prune_constraints_inf_norm(m1, tol=1e-4))
print(f"Number of support constraints: {s_N}")

Number of support constraints: 0


#### nonconvex SOP PAC bound
Consider Equation (7) in https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8299432

#### comparison of $\beta,\alpha$, and $\mathcal{N}$.

In [24]:
# as alternative, consider eqn 7 in the paper
def epsil(k,N1,beta1):
    res = (beta1/(N1*comb(N1, k)))**(1/(N1-k))
    return 1-res

In [25]:
beta = 10**-6
# alpha_sN = epsil(s_N, N, beta)
alpha_sN = epsil(1, 1000, beta)

print(f"With confidence of {(1-beta)*100:.6f}%, the non-violation of at least {1-alpha_sN:.6f}, and violation {alpha_sN:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.972720, and violation 0.027280


In [26]:
print(f"weaker bound: {(1-alpha_sN)*(1-beta)*100:.6f}%")

weaker bound: 97.271935%
